# 05 — Temporal Transformer Baseline

This notebook trains the **temporal Transformer comparator** for the Praxis pipeline using the frozen outputs from Notebooks 03 and 04.

## Primary objective

Estimate the predictive performance and inference latency of a causal, fixed-length temporal Transformer using the same event-level target and chronological partitions frozen by Notebook 04. The primary model uses the **fused physical + cyber feature set**. Optional physical-only and cyber-only temporal ablations can be enabled through the configuration cell.

## Modeling unit and sequence definition

- One prediction is produced for each candidate low-altitude point event.
- A sequence ends at the event being scored and contains that event plus up to the preceding `SEQUENCE_LENGTH - 1` site events.
- Sequences are left-padded when less history is available.
- A causal attention mask prevents earlier tokens from attending to later tokens.
- Validation and test sequences may use predictor data from earlier chronological events, including earlier partitions, because those observations would be available operationally. They never use future events or target labels as inputs.
- The model is evaluated with the exact `model_target` and split assignments written by Notebook 04.

## Interpretation boundary

The NF3 relationship is a controlled cross-domain alignment. Results do **not** establish that an aircraft event caused, or occurred simultaneously with, a real cyberattack at PANC. Model latency is measured after all required predictor features for the event window are available.

In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import platform
import random
import re
import sys
import time
import warnings
from dataclasses import dataclass, asdict
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Sequence

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Notebook 05 requires PyTorch. Install it in the active Jupyter environment "
        "before running this notebook."
    ) from exc

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

# -----------------------------------------------------------------------------
# Reproducibility and model configuration
# Environment variables make smoke tests and resource-constrained runs possible
# without editing the notebook source.
# -----------------------------------------------------------------------------
RANDOM_STATE = int(os.environ.get("TT_RANDOM_STATE", "42"))
MODEL_VERSION = "temporal_transformer_leakage_safe_v2"
PRIMARY_FEATURE_SET = "fused"
FEATURE_SETS_TO_RUN = [
    value.strip()
    for value in os.environ.get("TT_FEATURE_SETS", "fused").split(",")
    if value.strip()
]

SEQUENCE_LENGTH = int(os.environ.get("TT_SEQUENCE_LENGTH", "32"))
D_MODEL = int(os.environ.get("TT_D_MODEL", "64"))
N_HEADS = int(os.environ.get("TT_N_HEADS", "4"))
N_LAYERS = int(os.environ.get("TT_N_LAYERS", "2"))
FEEDFORWARD_DIM = int(os.environ.get("TT_FEEDFORWARD_DIM", "128"))
DROPOUT = float(os.environ.get("TT_DROPOUT", "0.15"))
BATCH_SIZE = int(os.environ.get("TT_BATCH_SIZE", "128"))
MAX_EPOCHS = int(os.environ.get("TT_MAX_EPOCHS", "30"))
EARLY_STOPPING_PATIENCE = int(os.environ.get("TT_PATIENCE", "5"))
LEARNING_RATE = float(os.environ.get("TT_LEARNING_RATE", "0.001"))
WEIGHT_DECAY = float(os.environ.get("TT_WEIGHT_DECAY", "0.0001"))
GRADIENT_CLIP_NORM = float(os.environ.get("TT_GRADIENT_CLIP_NORM", "1.0"))

ATTRIBUTION_SAMPLE_MAX = int(os.environ.get("TT_ATTRIBUTION_SAMPLE_MAX", "128"))
ATTENTION_SAMPLE_MAX = int(os.environ.get("TT_ATTENTION_SAMPLE_MAX", "256"))
LATENCY_EVENT_SAMPLE_MAX = int(os.environ.get("TT_LATENCY_EVENT_SAMPLE_MAX", "100"))
LATENCY_WARMUP_RUNS = int(os.environ.get("TT_LATENCY_WARMUP_RUNS", "10"))

if SEQUENCE_LENGTH < 2:
    raise ValueError("SEQUENCE_LENGTH must be at least 2.")
if D_MODEL % N_HEADS != 0:
    raise ValueError("D_MODEL must be divisible by N_HEADS.")
if not FEATURE_SETS_TO_RUN:
    raise ValueError("At least one feature set must be selected.")

PROJECT_ROOT = Path.cwd()
ALIGNMENT_DIR = PROJECT_ROOT / "data" / "processed" / "point_event_v2" / "nf3_alignment_v2"
MODELING_DIR = PROJECT_ROOT / "data" / "processed" / "point_event_v2" / "modeling_v4"
FEATURES_IN = ALIGNMENT_DIR / "candidate_point_events_v2_with_context_and_nf3_primary.csv"
SPLITS_IN = MODELING_DIR / "model_split_assignments_v4.csv"
FEATURE_MANIFEST_IN = MODELING_DIR / "feature_manifest_v4.json"
TARGET_DEFINITION_IN = MODELING_DIR / "model_target_definition_v4.json"
TABULAR_RESULTS_IN = MODELING_DIR / "baseline_model_results_v4.csv"
ALIGNMENT_SUMMARY_IN = ALIGNMENT_DIR / "point_event_nf3_alignment_leakage_safe_v2_build_summary.json"

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "point_event_v2" / "transformer_v2"
MODELS_DIR = OUTPUT_DIR / "models"
PREPROCESSING_DIR = OUTPUT_DIR / "preprocessing"
FIGURES_DIR = OUTPUT_DIR / "figures"

RESULTS_OUT = OUTPUT_DIR / "temporal_transformer_results_v2.csv"
PREDICTIONS_OUT = OUTPUT_DIR / "temporal_transformer_predictions_v2.csv"
THRESHOLDS_OUT = OUTPUT_DIR / "temporal_transformer_thresholds_v2.json"
HISTORY_OUT = OUTPUT_DIR / "temporal_transformer_training_history_v2.csv"
LATENCY_OUT = OUTPUT_DIR / "temporal_transformer_latency_v2.csv"
SEQUENCE_MANIFEST_OUT = OUTPUT_DIR / "temporal_transformer_sequence_manifest_v2.csv"
FEATURE_ATTRIBUTION_OUT = OUTPUT_DIR / "temporal_transformer_feature_attribution_v2.csv"
ATTENTION_SUMMARY_OUT = OUTPUT_DIR / "temporal_transformer_attention_summary_v2.csv"
COMPARISON_OUT = OUTPUT_DIR / "transformer_vs_tabular_v2.csv"
REFERENCE_OUT = OUTPUT_DIR / "transformer_reference_v2.json"
CONFIG_OUT = OUTPUT_DIR / "temporal_transformer_config_v2.json"
SUMMARY_OUT = OUTPUT_DIR / "temporal_transformer_build_summary_v2.json"
DATA_DICTIONARY_OUT = OUTPUT_DIR / "temporal_transformer_data_dictionary_v2.csv"
CHECKSUMS_OUT = OUTPUT_DIR / "SHA256SUMS.txt"

for directory in [OUTPUT_DIR, MODELS_DIR, PREPROCESSING_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

print(f"Project root:       {PROJECT_ROOT}")
print(f"Feature input:      {FEATURES_IN}")
print(f"Split input:        {SPLITS_IN}")
print(f"Feature sets:       {FEATURE_SETS_TO_RUN}")
print(f"PyTorch device:     {DEVICE}")
print(f"Output directory:   {OUTPUT_DIR}")

## 1. Validation, conversion, and metric helpers

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except (AttributeError, RuntimeError):
        pass


seed_everything(RANDOM_STATE)


def require_text(value: object, field_name: str) -> str:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")
    text = value.decode("utf-8", errors="strict") if isinstance(value, bytes) else str(value)
    text = text.strip()
    if not text or text.lower() in {"nan", "nat", "none", "<na>"}:
        raise ValueError(f"{field_name} cannot be empty or missing.")
    return text


def require_int(value: object, field_name: str) -> int:
    text = require_text(value, field_name)
    numeric = float(text)
    if not math.isfinite(numeric) or not numeric.is_integer():
        raise ValueError(f"{field_name} must be a finite whole number; received {value!r}.")
    return int(numeric)


def require_float(value: object, field_name: str) -> float:
    text = require_text(value, field_name)
    numeric = float(text)
    if not math.isfinite(numeric):
        raise ValueError(f"{field_name} must be finite; received {value!r}.")
    return numeric


def optional_float(value: object) -> float:
    try:
        return require_float(value, "optional numeric value")
    except (TypeError, ValueError):
        return float("nan")


def require_utc_timestamp(value: object, field_name: str) -> pd.Timestamp:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")
    if isinstance(value, pd.Timestamp):
        timestamp = value
    elif isinstance(value, (datetime, date, np.datetime64, str, int, float)):
        timestamp = pd.Timestamp(value)
    elif isinstance(value, bytes):
        timestamp = pd.Timestamp(value.decode("utf-8", errors="strict"))
    else:
        timestamp = pd.Timestamp(str(value))
    if pd.isna(timestamp):
        raise ValueError(f"{field_name} cannot be missing.")
    if timestamp.tzinfo is None:
        timestamp = timestamp.tz_localize("UTC")
    else:
        timestamp = timestamp.tz_convert("UTC")
    return timestamp


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def slugify(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


def has_both_classes(values: Sequence[int] | pd.Series | np.ndarray) -> bool:
    observed = set(
        pd.to_numeric(pd.Series(values), errors="coerce")
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    return observed == {0, 1}


def make_dense_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=2)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def safe_auprc(y_true: np.ndarray, probability: np.ndarray) -> float:
    if not has_both_classes(y_true):
        return float("nan")
    return float(average_precision_score(y_true, probability))


def safe_roc_auc(y_true: np.ndarray, probability: np.ndarray) -> float:
    if not has_both_classes(y_true):
        return float("nan")
    return float(roc_auc_score(y_true, probability))


def select_f1_threshold(y_true: np.ndarray, probability: np.ndarray) -> dict[str, object]:
    if not has_both_classes(y_true):
        return {
            "threshold": 0.5,
            "policy": "fixed_0.5_validation_single_class",
            "validation_f1": float("nan"),
        }
    precision_values, recall_values, thresholds = precision_recall_curve(y_true, probability)
    if thresholds.size == 0:
        return {"threshold": 0.5, "policy": "fixed_0.5_no_candidate_threshold", "validation_f1": 0.0}
    f1_values = 2.0 * precision_values[:-1] * recall_values[:-1] / np.maximum(
        precision_values[:-1] + recall_values[:-1], 1e-12
    )
    best_position = int(np.nanargmax(f1_values))
    return {
        "threshold": float(thresholds[best_position]),
        "policy": "maximize_validation_f1",
        "validation_f1": float(f1_values[best_position]),
    }


def classification_metrics(
    y_true: np.ndarray,
    probability: np.ndarray,
    threshold: float,
) -> dict[str, float | int]:
    predicted = (probability >= threshold).astype(np.int8)
    matrix = confusion_matrix(y_true, predicted, labels=[0, 1])
    true_negative, false_positive, false_negative, true_positive = [int(value) for value in matrix.ravel()]
    return {
        "auprc": safe_auprc(y_true, probability),
        "roc_auc": safe_roc_auc(y_true, probability),
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "balanced_accuracy": (
            float(balanced_accuracy_score(y_true, predicted))
            if has_both_classes(y_true)
            else float("nan")
        ),
        "brier_score": float(brier_score_loss(y_true, probability)),
        "true_negative": true_negative,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_positive": true_positive,
    }


def transformed_feature_names(preprocessor: ColumnTransformer) -> list[str]:
    try:
        return [str(value) for value in preprocessor.get_feature_names_out().tolist()]
    except (AttributeError, ValueError):
        names: list[str] = []
        for transformer_name, transformer, columns in preprocessor.transformers_:
            if transformer_name == "remainder" or transformer == "drop":
                continue
            column_names = [str(value) for value in list(columns)]
            if isinstance(transformer, Pipeline) and "onehot" in transformer.named_steps:
                encoder = transformer.named_steps["onehot"]
                try:
                    names.extend([str(value) for value in encoder.get_feature_names_out(column_names)])
                except AttributeError:
                    names.extend(column_names)
            else:
                names.extend(column_names)
        return names


def balanced_sample_positions(
    labels: np.ndarray,
    maximum_rows: int,
    random_state: int,
) -> np.ndarray:
    rng = np.random.default_rng(random_state)
    all_positions = np.arange(len(labels), dtype=np.int64)
    if len(all_positions) <= maximum_rows:
        return all_positions
    positive = all_positions[labels == 1]
    negative = all_positions[labels == 0]
    half = maximum_rows // 2
    selected_positive = rng.choice(positive, size=min(len(positive), half), replace=False) if len(positive) else np.array([], dtype=np.int64)
    remaining = maximum_rows - len(selected_positive)
    selected_negative = rng.choice(negative, size=min(len(negative), remaining), replace=False) if len(negative) else np.array([], dtype=np.int64)
    selected = np.concatenate([selected_positive, selected_negative])
    if len(selected) < maximum_rows:
        remaining_pool = np.setdiff1d(all_positions, selected, assume_unique=False)
        additional = rng.choice(remaining_pool, size=min(len(remaining_pool), maximum_rows - len(selected)), replace=False)
        selected = np.concatenate([selected, additional])
    rng.shuffle(selected)
    return selected.astype(np.int64)

## 2. Load and validate Notebook 03/04 artifacts

In [ ]:
required_paths = [FEATURES_IN, SPLITS_IN, FEATURE_MANIFEST_IN, TARGET_DEFINITION_IN]
for required_path in required_paths:
    if not required_path.is_file():
        raise FileNotFoundError(
            f"Required upstream artifact was not found: {required_path}\n"
            "Run the corrected Notebooks 03 and 04 successfully from the project root first."
        )

features = pd.read_csv(FEATURES_IN, low_memory=False)
splits = pd.read_csv(SPLITS_IN, low_memory=False)
feature_manifest = json.loads(FEATURE_MANIFEST_IN.read_text(encoding="utf-8"))
target_definition = json.loads(TARGET_DEFINITION_IN.read_text(encoding="utf-8"))

ID_COLUMN = "event_id"
TIME_COLUMN = "aircraft_event_time_utc"
GROUP_COLUMN = "aircraft_key"
TARGET_COLUMN = "model_target"
SPLIT_COLUMN = "chronological_split"
RAW_TARGET_COLUMN = "cyber_attack_window_label"

required_feature_columns = {ID_COLUMN, TIME_COLUMN, GROUP_COLUMN}
required_split_columns = {ID_COLUMN, TARGET_COLUMN, SPLIT_COLUMN}
missing_feature_columns = sorted(required_feature_columns - set(features.columns))
missing_split_columns = sorted(required_split_columns - set(splits.columns))
if missing_feature_columns:
    raise KeyError(f"Feature bank is missing required columns: {missing_feature_columns}")
if missing_split_columns:
    raise KeyError(f"Split assignment table is missing required columns: {missing_split_columns}")
if features[ID_COLUMN].duplicated().any():
    raise ValueError("Feature bank contains duplicate event_id values.")
if splits[ID_COLUMN].duplicated().any():
    raise ValueError("Split assignment table contains duplicate event_id values.")

split_columns_to_merge = [ID_COLUMN, TARGET_COLUMN, SPLIT_COLUMN]
for optional_column in [RAW_TARGET_COLUMN, "aircraft_group_split"]:
    if optional_column in splits.columns:
        split_columns_to_merge.append(optional_column)

model_data = features.merge(
    splits[split_columns_to_merge],
    on=ID_COLUMN,
    how="inner",
    validate="one_to_one",
)
if len(model_data) != len(features) or len(model_data) != len(splits):
    raise ValueError(
        "The feature bank and split assignments do not have a one-to-one event match: "
        f"features={len(features):,}, splits={len(splits):,}, merged={len(model_data):,}."
    )

model_data[ID_COLUMN] = model_data[ID_COLUMN].astype("string")
model_data[GROUP_COLUMN] = model_data[GROUP_COLUMN].astype("string")
model_data[TIME_COLUMN] = pd.to_datetime(model_data[TIME_COLUMN], utc=True, errors="raise")
model_data[TARGET_COLUMN] = pd.to_numeric(model_data[TARGET_COLUMN], errors="raise").astype("int8")
model_data[SPLIT_COLUMN] = model_data[SPLIT_COLUMN].astype("string")

if model_data[[ID_COLUMN, GROUP_COLUMN, TIME_COLUMN, TARGET_COLUMN, SPLIT_COLUMN]].isna().any().any():
    raise ValueError("Required identifiers, timestamps, targets, or split fields contain missing values.")
if not set(model_data[TARGET_COLUMN].unique().tolist()).issubset({0, 1}):
    raise ValueError("model_target must contain only 0 and 1.")
if set(model_data[SPLIT_COLUMN].unique().tolist()) != {"train", "validation", "test"}:
    raise ValueError(
        "chronological_split must contain exactly train, validation, and test. "
        f"Observed: {sorted(model_data[SPLIT_COLUMN].unique().tolist())}"
    )

model_data = model_data.sort_values([TIME_COLUMN, ID_COLUMN], kind="mergesort").reset_index(drop=True)

split_order = model_data[SPLIT_COLUMN].map({"train": 0, "validation": 1, "test": 2})
if split_order.isna().any() or not split_order.is_monotonic_increasing:
    raise ValueError(
        "Notebook 04 chronological assignments are not monotonic after sorting by event time."
    )
if not has_both_classes(model_data.loc[model_data[SPLIT_COLUMN].eq("train"), TARGET_COLUMN]):
    raise ValueError("The chronological training partition must contain both model-target classes.")

manifest_feature_sets = feature_manifest.get("feature_sets", {})
missing_requested_sets = sorted(set(FEATURE_SETS_TO_RUN) - set(manifest_feature_sets))
if missing_requested_sets:
    raise KeyError(
        f"Requested feature sets are absent from feature_manifest_v4.json: {missing_requested_sets}"
    )

print(f"Point events:          {len(model_data):,}")
print(f"Unique aircraft keys: {model_data[GROUP_COLUMN].nunique():,}")
print(f"Event range:           {model_data[TIME_COLUMN].min()} to {model_data[TIME_COLUMN].max()}")
print("\nChronological split and target distribution:")
display(
    model_data.groupby(SPLIT_COLUMN, sort=False)[TARGET_COLUMN]
    .agg(rows="size", positives="sum", prevalence="mean")
    .reset_index()
)
print("\nFrozen target definition:")
display(pd.DataFrame([target_definition]))

## 3. Freeze leakage-safe feature sets and fit training-only preprocessing

Notebook 05 reuses the feature sets written by Notebook 04. Numeric imputation/scaling and categorical imputation/one-hot encoding are fit **only** on chronological training rows. Absolute event timestamps, identifiers, labels, and target-derived fields remain excluded.

In [ ]:
excluded_target_columns = set(feature_manifest.get("excluded_target_derived_columns", []))
excluded_identifier_columns = set(feature_manifest.get("excluded_identifier_and_provenance_columns", []))
excluded_alignment_columns = set(feature_manifest.get("excluded_alignment_audit_columns", []))
for mandatory_exclusion in [TARGET_COLUMN, RAW_TARGET_COLUMN, ID_COLUMN, GROUP_COLUMN, TIME_COLUMN, SPLIT_COLUMN]:
    excluded_target_columns.add(mandatory_exclusion)


def existing_unique(columns: Iterable[str]) -> list[str]:
    output: list[str] = []
    seen: set[str] = set()
    for value in columns:
        column = str(value)
        if column in model_data.columns and column not in seen:
            output.append(column)
            seen.add(column)
    return output


def usable_columns_for_training(
    training_frame: pd.DataFrame,
    numeric_candidates: list[str],
    categorical_candidates: list[str],
) -> tuple[list[str], list[str], list[str]]:
    usable_numeric: list[str] = []
    usable_categorical: list[str] = []
    dropped: list[str] = []
    for column in numeric_candidates:
        numeric = pd.to_numeric(training_frame[column], errors="coerce")
        if numeric.notna().sum() == 0 or numeric.nunique(dropna=True) <= 1:
            dropped.append(column)
        else:
            usable_numeric.append(column)
    for column in categorical_candidates:
        categorical = training_frame[column].astype("string")
        if categorical.notna().sum() == 0 or categorical.nunique(dropna=True) <= 1:
            dropped.append(column)
        else:
            usable_categorical.append(column)
    return usable_numeric, usable_categorical, dropped


def build_preprocessor(numeric_columns: list[str], categorical_columns: list[str]) -> ColumnTransformer:
    transformers: list[tuple[str, Pipeline, list[str]]] = []
    if numeric_columns:
        numeric_pipeline = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                ("scaler", StandardScaler()),
            ]
        )
        transformers.append(("numeric", numeric_pipeline, numeric_columns))
    if categorical_columns:
        categorical_pipeline = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_dense_one_hot_encoder()),
            ]
        )
        transformers.append(("categorical", categorical_pipeline, categorical_columns))
    if not transformers:
        raise ValueError("No usable predictors remain for preprocessing.")
    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)


train_mask = model_data[SPLIT_COLUMN].eq("train")
prepared_feature_sets: dict[str, dict[str, Any]] = {}
feature_set_overview_rows: list[dict[str, object]] = []

for feature_set_name in FEATURE_SETS_TO_RUN:
    specification = manifest_feature_sets[feature_set_name]
    numeric_candidates = existing_unique(specification.get("numeric", []))
    categorical_candidates = existing_unique(specification.get("categorical", []))
    all_candidates = set(numeric_candidates + categorical_candidates)

    forbidden = sorted(
        all_candidates
        & (excluded_target_columns | excluded_identifier_columns | excluded_alignment_columns)
    )
    if forbidden:
        raise AssertionError(
            f"Forbidden fields entered the {feature_set_name} predictor set: {forbidden}"
        )

    for column in numeric_candidates:
        model_data[column] = pd.to_numeric(model_data[column], errors="coerce")
    for column in categorical_candidates:
        model_data[column] = model_data[column].astype("string")

    usable_numeric, usable_categorical, dropped = usable_columns_for_training(
        model_data.loc[train_mask],
        numeric_candidates,
        categorical_candidates,
    )
    usable_columns = usable_numeric + usable_categorical
    if not usable_columns:
        raise ValueError(f"No usable predictors remain for feature set {feature_set_name!r}.")

    preprocessor = build_preprocessor(usable_numeric, usable_categorical)
    preprocessor.fit(model_data.loc[train_mask, usable_columns])
    transformed = preprocessor.transform(model_data[usable_columns])
    feature_matrix = np.asarray(transformed, dtype=np.float32)
    if feature_matrix.ndim != 2 or feature_matrix.shape[0] != len(model_data):
        raise ValueError(
            f"Unexpected transformed shape for {feature_set_name}: {feature_matrix.shape}"
        )
    if not np.isfinite(feature_matrix).all():
        raise ValueError(f"Transformed feature matrix for {feature_set_name} contains non-finite values.")

    names = transformed_feature_names(preprocessor)
    if len(names) != feature_matrix.shape[1]:
        names = [f"transformed_feature_{position:04d}" for position in range(feature_matrix.shape[1])]

    prepared_feature_sets[feature_set_name] = {
        "numeric_columns": usable_numeric,
        "categorical_columns": usable_categorical,
        "dropped_columns": dropped,
        "raw_columns": usable_columns,
        "preprocessor": preprocessor,
        "feature_matrix": feature_matrix,
        "transformed_feature_names": names,
    }
    joblib.dump(preprocessor, PREPROCESSING_DIR / f"{slugify(feature_set_name)}_preprocessor.joblib")

    feature_set_overview_rows.append(
        {
            "feature_set": feature_set_name,
            "numeric_columns": len(usable_numeric),
            "categorical_columns": len(usable_categorical),
            "dropped_columns": len(dropped),
            "transformed_features": feature_matrix.shape[1],
        }
    )

display(pd.DataFrame(feature_set_overview_rows))

## 4. Build strictly backward-looking event sequences

The site-wide chronological event stream is used because the operational task scores each arriving event. Each sequence ends with the current event. A separate standardized log-time-gap channel represents irregular spacing between site events.

In [ ]:
event_times = model_data[TIME_COLUMN]
site_gap_seconds = event_times.diff().dt.total_seconds().fillna(0.0).clip(lower=0.0)
train_gap_log = np.log1p(site_gap_seconds.loc[train_mask].to_numpy(dtype=np.float64))
TIME_GAP_MEAN = float(np.mean(train_gap_log))
TIME_GAP_STD = float(np.std(train_gap_log))
if not math.isfinite(TIME_GAP_STD) or TIME_GAP_STD < 1e-8:
    TIME_GAP_STD = 1.0
all_gap_log = np.log1p(site_gap_seconds.to_numpy(dtype=np.float64))
standardized_time_gap = ((all_gap_log - TIME_GAP_MEAN) / TIME_GAP_STD).astype(np.float32)

row_count = len(model_data)
sequence_start_positions = np.maximum(0, np.arange(row_count, dtype=np.int64) - SEQUENCE_LENGTH + 1)
sequence_lengths = np.arange(row_count, dtype=np.int64) - sequence_start_positions + 1

sequence_manifest = pd.DataFrame(
    {
        ID_COLUMN: model_data[ID_COLUMN].astype("string"),
        SPLIT_COLUMN: model_data[SPLIT_COLUMN].astype("string"),
        "current_event_time_utc": model_data[TIME_COLUMN],
        "sequence_start_position": sequence_start_positions,
        "sequence_end_position": np.arange(row_count, dtype=np.int64),
        "sequence_length_available": sequence_lengths,
        "prior_context_event_count": sequence_lengths - 1,
        "sequence_left_padding_count": SEQUENCE_LENGTH - sequence_lengths,
        "sequence_truncated_flag": (sequence_start_positions > 0).astype(np.int8),
        TARGET_COLUMN: model_data[TARGET_COLUMN].astype("int8"),
    }
)
sequence_manifest["sequence_start_event_id"] = [
    require_text(
        model_data.iloc[require_int(position, "sequence start position")][ID_COLUMN],
        "sequence start event_id",
    )
    for position in sequence_start_positions
]
sequence_manifest["sequence_start_time_utc"] = [
    require_utc_timestamp(
        model_data.iloc[require_int(position, "sequence start position")][TIME_COLUMN],
        "sequence start time",
    )
    for position in sequence_start_positions
]
sequence_manifest.to_csv(SEQUENCE_MANIFEST_OUT, index=False)

if not (
    pd.to_datetime(sequence_manifest["sequence_start_time_utc"], utc=True)
    <= pd.to_datetime(sequence_manifest["current_event_time_utc"], utc=True)
).all():
    raise AssertionError("At least one sequence starts after its current event.")
if not (sequence_manifest["sequence_end_position"] == np.arange(row_count)).all():
    raise AssertionError("At least one sequence does not end at its current event.")

print(f"Sequence length:          {SEQUENCE_LENGTH}")
print(f"Median available length: {float(np.median(sequence_lengths)):.1f}")
print(f"Fully populated rows:    {int((sequence_lengths == SEQUENCE_LENGTH).sum()):,}")
print(f"Time-gap mean (log1p):   {TIME_GAP_MEAN:.6f}")
print(f"Time-gap std (log1p):    {TIME_GAP_STD:.6f}")

## 5. Dataset and temporal Transformer architecture

In [ ]:
class TemporalSequenceDataset(Dataset):
    def __init__(
        self,
        feature_matrix: np.ndarray,
        time_gap_values: np.ndarray,
        labels: np.ndarray,
        row_positions: np.ndarray,
        sequence_length: int,
    ) -> None:
        self.feature_matrix = feature_matrix
        self.time_gap_values = time_gap_values
        self.labels = labels.astype(np.float32, copy=False)
        self.row_positions = row_positions.astype(np.int64, copy=False)
        self.sequence_length = int(sequence_length)

    def __len__(self) -> int:
        return int(len(self.row_positions))

    def __getitem__(self, item: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        row_position = int(self.row_positions[item])
        start_position = max(0, row_position - self.sequence_length + 1)
        available = row_position - start_position + 1
        left_padding = self.sequence_length - available

        feature_count = int(self.feature_matrix.shape[1])
        sequence_features = np.zeros((self.sequence_length, feature_count), dtype=np.float32)
        sequence_time = np.zeros((self.sequence_length, 1), dtype=np.float32)
        padding_mask = np.ones(self.sequence_length, dtype=np.bool_)

        sequence_features[left_padding:] = self.feature_matrix[start_position : row_position + 1]
        sequence_time[left_padding:, 0] = self.time_gap_values[start_position : row_position + 1]
        padding_mask[left_padding:] = False

        return (
            torch.from_numpy(sequence_features),
            torch.from_numpy(sequence_time),
            torch.from_numpy(padding_mask),
            torch.tensor(self.labels[row_position], dtype=torch.float32),
            torch.tensor(row_position, dtype=torch.int64),
        )


class TemporalTransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, feedforward_dim: int, dropout: float) -> None:
        super().__init__()
        self.norm_attention = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.dropout_attention = nn.Dropout(dropout)
        self.norm_feedforward = nn.LayerNorm(d_model)
        self.feedforward = nn.Sequential(
            nn.Linear(d_model, feedforward_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feedforward_dim, d_model),
            nn.Dropout(dropout),
        )

    def forward(
        self,
        inputs: torch.Tensor,
        causal_mask: torch.Tensor,
        padding_mask: torch.Tensor,
        return_attention: bool = False,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        normalized = self.norm_attention(inputs)
        attention_output, attention_weights = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=causal_mask,
            key_padding_mask=padding_mask,
            need_weights=return_attention,
            average_attn_weights=False,
        )
        hidden = inputs + self.dropout_attention(attention_output)
        hidden = hidden + self.feedforward(self.norm_feedforward(hidden))
        return hidden, attention_weights if return_attention else None


class TemporalTransformerClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int,
        sequence_length: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        feedforward_dim: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.sequence_length = int(sequence_length)
        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.time_projection = nn.Sequential(nn.Linear(1, d_model), nn.Tanh())
        self.position_embedding = nn.Embedding(sequence_length, d_model)
        self.layers = nn.ModuleList(
            [
                TemporalTransformerBlock(d_model, n_heads, feedforward_dim, dropout)
                for _ in range(n_layers)
            ]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, max(16, d_model // 2)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(max(16, d_model // 2), 1),
        )

    def forward(
        self,
        sequence_features: torch.Tensor,
        sequence_time: torch.Tensor,
        padding_mask: torch.Tensor,
        return_attention: bool = False,
    ) -> tuple[torch.Tensor, list[torch.Tensor]]:
        batch_size, sequence_length, _ = sequence_features.shape
        if sequence_length != self.sequence_length:
            raise ValueError(
                f"Expected sequence length {self.sequence_length}; received {sequence_length}."
            )
        positions = torch.arange(sequence_length, device=sequence_features.device)
        hidden = (
            self.input_projection(sequence_features)
            + self.time_projection(sequence_time)
            + self.position_embedding(positions).unsqueeze(0).expand(batch_size, -1, -1)
        )
        causal_mask = torch.triu(
            torch.ones(sequence_length, sequence_length, device=sequence_features.device, dtype=torch.bool),
            diagonal=1,
        )
        collected_attention: list[torch.Tensor] = []
        for layer in self.layers:
            hidden, attention = layer(
                hidden,
                causal_mask=causal_mask,
                padding_mask=padding_mask,
                return_attention=return_attention,
            )
            if attention is not None:
                collected_attention.append(attention)
        final_token = self.final_norm(hidden)[:, -1, :]
        logits = self.classifier(final_token).squeeze(-1)
        return logits, collected_attention


@dataclass(frozen=True)
class TrainingConfiguration:
    sequence_length: int
    d_model: int
    n_heads: int
    n_layers: int
    feedforward_dim: int
    dropout: float
    batch_size: int
    maximum_epochs: int
    patience: int
    learning_rate: float
    weight_decay: float
    gradient_clip_norm: float


training_configuration = TrainingConfiguration(
    sequence_length=SEQUENCE_LENGTH,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    feedforward_dim=FEEDFORWARD_DIM,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    maximum_epochs=MAX_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    gradient_clip_norm=GRADIENT_CLIP_NORM,
)
print(training_configuration)

## 6. Training, prediction, and latency helpers

In [ ]:
def make_loader(
    dataset: TemporalSequenceDataset,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(RANDOM_STATE)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
        generator=generator if shuffle else None,
    )


def predict_dataset(
    model: TemporalTransformerClassifier,
    loader: DataLoader,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    probabilities: list[np.ndarray] = []
    labels: list[np.ndarray] = []
    positions: list[np.ndarray] = []
    with torch.no_grad():
        for sequence_features, sequence_time, padding_mask, batch_labels, batch_positions in loader:
            sequence_features = sequence_features.to(DEVICE)
            sequence_time = sequence_time.to(DEVICE)
            padding_mask = padding_mask.to(DEVICE)
            logits, _ = model(sequence_features, sequence_time, padding_mask)
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
            labels.append(batch_labels.numpy())
            positions.append(batch_positions.numpy())
    return (
        np.concatenate(probabilities).astype(np.float64),
        np.concatenate(labels).astype(np.int8),
        np.concatenate(positions).astype(np.int64),
    )


def validation_score(y_true: np.ndarray, probability: np.ndarray, mean_loss: float) -> tuple[float, str]:
    if has_both_classes(y_true):
        return safe_auprc(y_true, probability), "validation_auprc"
    return -float(mean_loss), "negative_validation_loss"


def train_temporal_transformer(
    feature_matrix: np.ndarray,
    feature_set_name: str,
) -> tuple[
    TemporalTransformerClassifier,
    pd.DataFrame,
    dict[str, np.ndarray],
    dict[str, object],
]:
    labels = model_data[TARGET_COLUMN].to_numpy(dtype=np.int8)
    split_values = model_data[SPLIT_COLUMN].astype(str).to_numpy()
    row_positions = np.arange(len(model_data), dtype=np.int64)

    split_positions = {
        split_name: row_positions[split_values == split_name]
        for split_name in ["train", "validation", "test"]
    }
    datasets = {
        split_name: TemporalSequenceDataset(
            feature_matrix,
            standardized_time_gap,
            labels,
            positions,
            SEQUENCE_LENGTH,
        )
        for split_name, positions in split_positions.items()
    }
    loaders = {
        "train": make_loader(datasets["train"], BATCH_SIZE, shuffle=True),
        "validation": make_loader(datasets["validation"], BATCH_SIZE, shuffle=False),
        "test": make_loader(datasets["test"], BATCH_SIZE, shuffle=False),
    }

    model = TemporalTransformerClassifier(
        input_dim=int(feature_matrix.shape[1]),
        sequence_length=SEQUENCE_LENGTH,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        feedforward_dim=FEEDFORWARD_DIM,
        dropout=DROPOUT,
    ).to(DEVICE)

    y_train = labels[split_positions["train"]]
    positive_count = int((y_train == 1).sum())
    negative_count = int((y_train == 0).sum())
    if positive_count == 0 or negative_count == 0:
        raise ValueError("Chronological training must contain both target classes.")
    positive_weight = float(negative_count / positive_count)
    loss_function = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(positive_weight, dtype=torch.float32, device=DEVICE)
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state: dict[str, torch.Tensor] | None = None
    best_score = -float("inf")
    best_epoch = 0
    score_name = "validation_auprc"
    epochs_without_improvement = 0
    history_rows: list[dict[str, object]] = []
    training_start = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_rows = 0
        for sequence_features, sequence_time, padding_mask, batch_labels, _ in loaders["train"]:
            sequence_features = sequence_features.to(DEVICE)
            sequence_time = sequence_time.to(DEVICE)
            padding_mask = padding_mask.to(DEVICE)
            batch_labels = batch_labels.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(sequence_features, sequence_time, padding_mask)
            loss = loss_function(logits, batch_labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
            optimizer.step()

            batch_rows = int(batch_labels.shape[0])
            train_loss_sum += float(loss.detach().cpu().item()) * batch_rows
            train_rows += batch_rows

        mean_train_loss = train_loss_sum / max(train_rows, 1)

        model.eval()
        validation_loss_sum = 0.0
        validation_rows = 0
        validation_probabilities: list[np.ndarray] = []
        validation_labels: list[np.ndarray] = []
        with torch.no_grad():
            for sequence_features, sequence_time, padding_mask, batch_labels, _ in loaders["validation"]:
                sequence_features = sequence_features.to(DEVICE)
                sequence_time = sequence_time.to(DEVICE)
                padding_mask = padding_mask.to(DEVICE)
                batch_labels_device = batch_labels.to(DEVICE)
                logits, _ = model(sequence_features, sequence_time, padding_mask)
                loss = loss_function(logits, batch_labels_device)
                batch_rows = int(batch_labels.shape[0])
                validation_loss_sum += float(loss.detach().cpu().item()) * batch_rows
                validation_rows += batch_rows
                validation_probabilities.append(torch.sigmoid(logits).cpu().numpy())
                validation_labels.append(batch_labels.numpy())

        mean_validation_loss = validation_loss_sum / max(validation_rows, 1)
        validation_probability = np.concatenate(validation_probabilities).astype(np.float64)
        validation_y = np.concatenate(validation_labels).astype(np.int8)
        current_score, score_name = validation_score(
            validation_y,
            validation_probability,
            mean_validation_loss,
        )
        current_auprc = safe_auprc(validation_y, validation_probability)

        history_rows.append(
            {
                "model_version": MODEL_VERSION,
                "feature_set": feature_set_name,
                "epoch": epoch,
                "train_loss": mean_train_loss,
                "validation_loss": mean_validation_loss,
                "validation_auprc": current_auprc,
                "selection_score": current_score,
                "selection_metric": score_name,
            }
        )

        improved = current_score > best_score + 1e-8
        if improved:
            best_score = current_score
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"{feature_set_name} epoch {epoch:02d}: "
            f"train_loss={mean_train_loss:.5f}, val_loss={mean_validation_loss:.5f}, "
            f"val_auprc={current_auprc:.5f}, best_epoch={best_epoch}"
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping {feature_set_name} after epoch {epoch}.")
            break

    if best_state is None:
        raise RuntimeError(f"No valid checkpoint was produced for {feature_set_name}.")
    model.load_state_dict(best_state)
    fit_seconds = float(time.perf_counter() - training_start)

    predictions: dict[str, np.ndarray] = {}
    for split_name in ["validation", "test"]:
        probability, y_values, positions = predict_dataset(model, loaders[split_name])
        predictions[f"{split_name}_probability"] = probability
        predictions[f"{split_name}_target"] = y_values
        predictions[f"{split_name}_position"] = positions

    training_metadata: dict[str, object] = {
        "best_epoch": best_epoch,
        "best_selection_score": best_score,
        "selection_metric": score_name,
        "fit_seconds": fit_seconds,
        "positive_weight": positive_weight,
        "train_positive_rows": positive_count,
        "train_negative_rows": negative_count,
        "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
    }
    return model, pd.DataFrame(history_rows), predictions, training_metadata


def build_single_event_tensors(
    row_position: int,
    raw_columns: list[str],
    preprocessor: ColumnTransformer,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    start_position = max(0, row_position - SEQUENCE_LENGTH + 1)
    raw_sequence = model_data.loc[start_position:row_position, raw_columns]
    transformed = np.asarray(preprocessor.transform(raw_sequence), dtype=np.float32)
    available = int(transformed.shape[0])
    left_padding = SEQUENCE_LENGTH - available
    feature_count = int(transformed.shape[1])

    sequence_features = np.zeros((SEQUENCE_LENGTH, feature_count), dtype=np.float32)
    sequence_time = np.zeros((SEQUENCE_LENGTH, 1), dtype=np.float32)
    padding_mask = np.ones(SEQUENCE_LENGTH, dtype=np.bool_)
    sequence_features[left_padding:] = transformed
    sequence_time[left_padding:, 0] = standardized_time_gap[start_position : row_position + 1]
    padding_mask[left_padding:] = False

    return (
        torch.from_numpy(sequence_features).unsqueeze(0).to(DEVICE),
        torch.from_numpy(sequence_time).unsqueeze(0).to(DEVICE),
        torch.from_numpy(padding_mask).unsqueeze(0).to(DEVICE),
    )


def measure_end_to_end_latency(
    model: TemporalTransformerClassifier,
    raw_columns: list[str],
    preprocessor: ColumnTransformer,
    candidate_positions: np.ndarray,
) -> dict[str, float | int]:
    if len(candidate_positions) == 0:
        return {
            "latency_events": 0,
            "latency_mean_ms": float("nan"),
            "latency_p50_ms": float("nan"),
            "latency_p95_ms": float("nan"),
        }
    rng = np.random.default_rng(RANDOM_STATE)
    selected = candidate_positions
    if len(selected) > LATENCY_EVENT_SAMPLE_MAX:
        selected = rng.choice(selected, size=LATENCY_EVENT_SAMPLE_MAX, replace=False)

    model.eval()
    warmup_position = int(selected[0])
    for _ in range(LATENCY_WARMUP_RUNS):
        tensors = build_single_event_tensors(warmup_position, raw_columns, preprocessor)
        with torch.no_grad():
            model(*tensors)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    measurements: list[float] = []
    for position_value in selected:
        position = int(position_value)
        start = time.perf_counter()
        tensors = build_single_event_tensors(position, raw_columns, preprocessor)
        with torch.no_grad():
            model(*tensors)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        measurements.append((time.perf_counter() - start) * 1000.0)

    values = np.asarray(measurements, dtype=np.float64)
    return {
        "latency_events": int(len(values)),
        "latency_mean_ms": float(np.mean(values)),
        "latency_p50_ms": float(np.percentile(values, 50)),
        "latency_p95_ms": float(np.percentile(values, 95)),
    }

## 7. Train and evaluate the temporal Transformer

In [ ]:
results_rows: list[dict[str, object]] = []
prediction_frames: list[pd.DataFrame] = []
history_frames: list[pd.DataFrame] = []
latency_rows: list[dict[str, object]] = []
threshold_records: dict[str, dict[str, object]] = {}
trained_models: dict[str, TemporalTransformerClassifier] = {}
training_metadata_by_set: dict[str, dict[str, object]] = {}

for feature_set_name in FEATURE_SETS_TO_RUN:
    prepared = prepared_feature_sets[feature_set_name]
    feature_matrix = prepared["feature_matrix"]
    print(f"\nTraining temporal Transformer for feature set: {feature_set_name}")

    model, history, predictions, training_metadata = train_temporal_transformer(
        feature_matrix,
        feature_set_name,
    )
    trained_models[feature_set_name] = model
    training_metadata_by_set[feature_set_name] = training_metadata
    history_frames.append(history)

    validation_target = predictions["validation_target"]
    validation_probability = predictions["validation_probability"]
    threshold_selection = select_f1_threshold(validation_target, validation_probability)
    selected_threshold = require_float(threshold_selection["threshold"], "selected threshold")
    threshold_records[feature_set_name] = threshold_selection

    test_target = predictions["test_target"]
    test_probability = predictions["test_probability"]
    validation_metrics = classification_metrics(
        validation_target,
        validation_probability,
        selected_threshold,
    )
    test_metrics = classification_metrics(
        test_target,
        test_probability,
        selected_threshold,
    )

    test_positions = predictions["test_position"]
    latency = measure_end_to_end_latency(
        model,
        prepared["raw_columns"],
        prepared["preprocessor"],
        test_positions,
    )
    latency_rows.append(
        {
            "model_version": MODEL_VERSION,
            "feature_set": feature_set_name,
            "measurement_type": "raw_sequence_preprocessing_plus_model_forward",
            **latency,
        }
    )

    checkpoint_path = MODELS_DIR / f"{slugify(feature_set_name)}_temporal_transformer.pt"
    torch.save(
        {
            "model_version": MODEL_VERSION,
            "feature_set": feature_set_name,
            "state_dict": model.state_dict(),
            "architecture": {
                "input_dim": int(feature_matrix.shape[1]),
                **asdict(training_configuration),
            },
            "transformed_feature_names": prepared["transformed_feature_names"],
            "raw_numeric_columns": prepared["numeric_columns"],
            "raw_categorical_columns": prepared["categorical_columns"],
            "time_gap_mean_log1p": TIME_GAP_MEAN,
            "time_gap_std_log1p": TIME_GAP_STD,
            "target_definition": target_definition,
            "selected_threshold": selected_threshold,
            "training_metadata": training_metadata,
        },
        checkpoint_path,
    )

    result_row: dict[str, object] = {
        "model_version": MODEL_VERSION,
        "feature_set": feature_set_name,
        "model": "causal_temporal_transformer",
        "target_mode": target_definition.get("mode"),
        "target_source_column": target_definition.get("source_column"),
        "sequence_length": SEQUENCE_LENGTH,
        "input_transformed_features": int(feature_matrix.shape[1]),
        "selected_threshold": selected_threshold,
        "threshold_policy": threshold_selection["policy"],
        "best_epoch": training_metadata["best_epoch"],
        "fit_seconds": training_metadata["fit_seconds"],
        "parameter_count": training_metadata["parameter_count"],
        "positive_weight": training_metadata["positive_weight"],
        **{f"validation_{key}": value for key, value in validation_metrics.items()},
        **{f"test_{key}": value for key, value in test_metrics.items()},
        **latency,
    }
    results_rows.append(result_row)

    for split_name in ["validation", "test"]:
        positions = predictions[f"{split_name}_position"]
        probabilities = predictions[f"{split_name}_probability"]
        targets = predictions[f"{split_name}_target"]
        predicted_labels = (probabilities >= selected_threshold).astype(np.int8)
        prediction_frames.append(
            pd.DataFrame(
                {
                    "model_version": MODEL_VERSION,
                    "feature_set": feature_set_name,
                    "split": split_name,
                    ID_COLUMN: model_data.iloc[positions][ID_COLUMN].astype(str).to_numpy(),
                    TIME_COLUMN: model_data.iloc[positions][TIME_COLUMN].astype(str).to_numpy(),
                    GROUP_COLUMN: model_data.iloc[positions][GROUP_COLUMN].astype(str).to_numpy(),
                    TARGET_COLUMN: targets,
                    "predicted_probability": probabilities,
                    "selected_threshold": selected_threshold,
                    "predicted_label": predicted_labels,
                    "row_position": positions,
                }
            )
        )

results = pd.DataFrame(results_rows)
prediction_table = pd.concat(prediction_frames, ignore_index=True)
training_history = pd.concat(history_frames, ignore_index=True)
latency_table = pd.DataFrame(latency_rows)

results.to_csv(RESULTS_OUT, index=False)
prediction_table.to_csv(PREDICTIONS_OUT, index=False)
training_history.to_csv(HISTORY_OUT, index=False)
latency_table.to_csv(LATENCY_OUT, index=False)
THRESHOLDS_OUT.write_text(json.dumps(threshold_records, indent=2, default=str), encoding="utf-8")

display(results)
print(f"Saved predictions: {len(prediction_table):,}")

## 8. Diagnostic explanations

Two model diagnostics are produced for the primary fused Transformer:

1. **Gradient × input attribution** aggregates absolute sensitivity across selected test sequences. This is a local-gradient diagnostic, not a causal explanation.
2. **Final-layer attention summary** reports how much the final query token attends to each historical position. Attention weights describe model behavior but should not be treated as definitive feature importance.

In [ ]:
feature_attribution = pd.DataFrame()
attention_summary = pd.DataFrame()

if PRIMARY_FEATURE_SET in trained_models:
    primary_model = trained_models[PRIMARY_FEATURE_SET]
    primary_prepared = prepared_feature_sets[PRIMARY_FEATURE_SET]
    primary_matrix = primary_prepared["feature_matrix"]
    primary_names = primary_prepared["transformed_feature_names"]
    labels_all = model_data[TARGET_COLUMN].to_numpy(dtype=np.int8)
    test_global_positions = np.flatnonzero(model_data[SPLIT_COLUMN].eq("test").to_numpy())
    test_labels = labels_all[test_global_positions]

    selected_local_positions = balanced_sample_positions(
        test_labels,
        maximum_rows=ATTRIBUTION_SAMPLE_MAX,
        random_state=RANDOM_STATE,
    )
    selected_global_positions = test_global_positions[selected_local_positions]
    explanation_dataset = TemporalSequenceDataset(
        primary_matrix,
        standardized_time_gap,
        labels_all,
        selected_global_positions,
        SEQUENCE_LENGTH,
    )
    explanation_loader = make_loader(
        explanation_dataset,
        batch_size=min(BATCH_SIZE, max(1, len(explanation_dataset))),
        shuffle=False,
    )

    attribution_sum = np.zeros((SEQUENCE_LENGTH, primary_matrix.shape[1]), dtype=np.float64)
    attribution_rows = 0
    primary_model.eval()
    for sequence_features, sequence_time, padding_mask, _, _ in explanation_loader:
        sequence_features = sequence_features.to(DEVICE)
        sequence_features.requires_grad_(True)
        sequence_time = sequence_time.to(DEVICE)
        padding_mask = padding_mask.to(DEVICE)
        primary_model.zero_grad(set_to_none=True)
        logits, _ = primary_model(sequence_features, sequence_time, padding_mask)
        logits.sum().backward()
        gradients = sequence_features.grad
        if gradients is None:
            raise RuntimeError("Gradient attribution did not produce input gradients.")
        attribution = torch.abs(gradients * sequence_features).detach().cpu().numpy()
        valid_mask = (~padding_mask).detach().cpu().numpy()[..., None]
        attribution_sum += (attribution * valid_mask).sum(axis=0)
        attribution_rows += int(sequence_features.shape[0])

    mean_attribution_by_position = attribution_sum / max(attribution_rows, 1)
    overall_attribution = mean_attribution_by_position.mean(axis=0)
    current_attribution = mean_attribution_by_position[-1]
    history_attribution = mean_attribution_by_position[:-1].mean(axis=0)
    feature_attribution = pd.DataFrame(
        {
            "model_version": MODEL_VERSION,
            "feature_set": PRIMARY_FEATURE_SET,
            "transformed_feature": primary_names,
            "mean_absolute_gradient_x_input": overall_attribution,
            "current_token_attribution": current_attribution,
            "history_token_mean_attribution": history_attribution,
            "explanation_sequence_count": attribution_rows,
        }
    ).sort_values("mean_absolute_gradient_x_input", ascending=False, kind="mergesort")
    feature_attribution["importance_rank"] = np.arange(1, len(feature_attribution) + 1)
    feature_attribution.to_csv(FEATURE_ATTRIBUTION_OUT, index=False)

    attention_local_positions = balanced_sample_positions(
        test_labels,
        maximum_rows=ATTENTION_SAMPLE_MAX,
        random_state=RANDOM_STATE + 1,
    )
    attention_global_positions = test_global_positions[attention_local_positions]
    attention_dataset = TemporalSequenceDataset(
        primary_matrix,
        standardized_time_gap,
        labels_all,
        attention_global_positions,
        SEQUENCE_LENGTH,
    )
    attention_loader = make_loader(
        attention_dataset,
        batch_size=min(BATCH_SIZE, max(1, len(attention_dataset))),
        shuffle=False,
    )

    attention_records: list[dict[str, object]] = []
    primary_model.eval()
    with torch.no_grad():
        for sequence_features, sequence_time, padding_mask, batch_labels, _ in attention_loader:
            sequence_features = sequence_features.to(DEVICE)
            sequence_time = sequence_time.to(DEVICE)
            padding_mask_device = padding_mask.to(DEVICE)
            _, attention_layers = primary_model(
                sequence_features,
                sequence_time,
                padding_mask_device,
                return_attention=True,
            )
            if not attention_layers:
                continue
            final_attention = attention_layers[-1].detach().cpu().numpy()
            # [batch, heads, query, key] -> final query token, average heads.
            final_query_attention = final_attention[:, :, -1, :].mean(axis=1)
            padding_numpy = padding_mask.numpy()
            labels_numpy = batch_labels.numpy().astype(np.int8)
            for row_position in range(final_query_attention.shape[0]):
                for sequence_position in range(SEQUENCE_LENGTH):
                    if padding_numpy[row_position, sequence_position]:
                        continue
                    attention_records.append(
                        {
                            "target_class": int(labels_numpy[row_position]),
                            "sequence_position": sequence_position,
                            "steps_before_current": SEQUENCE_LENGTH - 1 - sequence_position,
                            "attention_weight": float(final_query_attention[row_position, sequence_position]),
                        }
                    )

    if attention_records:
        attention_detail = pd.DataFrame(attention_records)
        attention_summary = (
            attention_detail.groupby(["target_class", "sequence_position", "steps_before_current"], as_index=False)
            .agg(
                mean_attention_weight=("attention_weight", "mean"),
                median_attention_weight=("attention_weight", "median"),
                contributing_sequences=("attention_weight", "size"),
            )
            .sort_values(["target_class", "steps_before_current"], kind="mergesort")
        )
        attention_summary.insert(0, "model_version", MODEL_VERSION)
        attention_summary.insert(1, "feature_set", PRIMARY_FEATURE_SET)
        attention_summary.to_csv(ATTENTION_SUMMARY_OUT, index=False)

    display(feature_attribution.head(20))
    display(attention_summary.head(20))
else:
    warnings.warn(
        f"Primary feature set {PRIMARY_FEATURE_SET!r} was not trained; explanation outputs were skipped."
    )

## 9. Figures and comparison with the best tabular fused baseline

In [ ]:
for feature_set_name in FEATURE_SETS_TO_RUN:
    feature_history = training_history.loc[training_history["feature_set"].eq(feature_set_name)]
    if not feature_history.empty:
        plt.figure(figsize=(8, 5))
        plt.plot(feature_history["epoch"], feature_history["train_loss"], label="Training loss")
        plt.plot(feature_history["epoch"], feature_history["validation_loss"], label="Validation loss")
        plt.xlabel("Epoch")
        plt.ylabel("Weighted binary cross-entropy")
        plt.title(f"Temporal Transformer Training — {feature_set_name}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f"{slugify(feature_set_name)}_training_history.png", dpi=180)
        plt.close()

    feature_predictions = prediction_table.loc[
        prediction_table["feature_set"].eq(feature_set_name)
        & prediction_table["split"].eq("test")
    ]
    if not feature_predictions.empty and has_both_classes(feature_predictions[TARGET_COLUMN]):
        test_y = feature_predictions[TARGET_COLUMN].to_numpy(dtype=np.int8)
        test_probability = feature_predictions["predicted_probability"].to_numpy(dtype=np.float64)
        precision_values, recall_values, _ = precision_recall_curve(test_y, test_probability)
        prevalence = float(np.mean(test_y))
        plt.figure(figsize=(7, 5))
        plt.plot(recall_values, precision_values, label="Temporal Transformer")
        plt.axhline(prevalence, linestyle="--", label=f"Prevalence = {prevalence:.3f}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"Test Precision–Recall Curve — {feature_set_name}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f"{slugify(feature_set_name)}_test_precision_recall.png", dpi=180)
        plt.close()

        threshold = require_float(
            results.loc[results["feature_set"].eq(feature_set_name), "selected_threshold"].iloc[0],
            "figure threshold",
        )
        predicted = (test_probability >= threshold).astype(np.int8)
        matrix = confusion_matrix(test_y, predicted, labels=[0, 1])
        plt.figure(figsize=(5, 4))
        plt.imshow(matrix)
        plt.xticks([0, 1], ["Predicted 0", "Predicted 1"])
        plt.yticks([0, 1], ["Actual 0", "Actual 1"])
        for row in range(2):
            for column in range(2):
                plt.text(column, row, str(int(matrix[row, column])), ha="center", va="center")
        plt.title(f"Test Confusion Matrix — {feature_set_name}")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f"{slugify(feature_set_name)}_test_confusion_matrix.png", dpi=180)
        plt.close()

if not feature_attribution.empty:    
    top_attribution = feature_attribution.head(20).sort_values(
        "mean_absolute_gradient_x_input", ascending=True
    )
    plt.figure(figsize=(10, 7))
    plt.barh(top_attribution["transformed_feature"], top_attribution["mean_absolute_gradient_x_input"])
    plt.xlabel("Mean absolute gradient × input")
    plt.title("Temporal Transformer Feature Attribution — Fused")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fused_feature_attribution.png", dpi=180)
    plt.close()

if not attention_summary.empty:
    attention_summary["target_class"] = pd.to_numeric(
        attention_summary["target_class"],
        errors="raise",
    ).astype("int8")

    plt.figure(figsize=(8, 5))

    for target_class, class_frame in attention_summary.groupby(
        "target_class",
        sort=True,
    ):
        normalized_target_class = require_int(
            target_class,
            "attention target class",
        )

        ordered = class_frame.sort_values(
            "steps_before_current"
        )

        plt.plot(
            ordered["steps_before_current"],
            ordered["mean_attention_weight"],
            marker="o",
            label=f"Target {normalized_target_class}",
        )

    plt.xlabel("Steps before current event")
    plt.ylabel("Mean final-token attention")
    plt.title("Final-Layer Attention by Temporal Position")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / "fused_attention_by_position.png",
        dpi=180,
    )
    plt.close()

comparison_rows: list[dict[str, object]] = []
if TABULAR_RESULTS_IN.is_file():
    tabular_results = pd.read_csv(TABULAR_RESULTS_IN, low_memory=False)
    required_tabular_columns = {"split_strategy", "feature_set", "model", "test_auprc"}
    if required_tabular_columns.issubset(tabular_results.columns):
        candidates = tabular_results.loc[
            tabular_results["split_strategy"].astype("string").eq("chronological_primary")
            & tabular_results["feature_set"].astype("string").eq("fused")
        ].copy()
        candidates["test_auprc"] = pd.to_numeric(candidates["test_auprc"], errors="coerce")
        candidates = candidates.dropna(subset=["test_auprc"])
        if not candidates.empty:
            best_row = candidates.sort_values("test_auprc", ascending=False).iloc[0]
            transformer_row = results.loc[results["feature_set"].eq(PRIMARY_FEATURE_SET)]
            if not transformer_row.empty:
                transformer_auprc = optional_float(transformer_row.iloc[0]["test_auprc"])
                tabular_auprc = optional_float(best_row["test_auprc"])
                comparison_rows.append(
                    {
                        "model_version": MODEL_VERSION,
                        "transformer_feature_set": PRIMARY_FEATURE_SET,
                        "transformer_test_auprc": transformer_auprc,
                        "best_tabular_model": require_text(best_row["model"], "best tabular model"),
                        "best_tabular_test_auprc": tabular_auprc,
                        "transformer_minus_tabular_auprc": transformer_auprc - tabular_auprc,
                    }
                )
    else:
        warnings.warn(
            f"Tabular results lack comparison columns: {sorted(required_tabular_columns - set(tabular_results.columns))}"
        )
else:
    warnings.warn("Notebook 04 result table was not found; tabular comparison was skipped.")

comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(COMPARISON_OUT, index=False)
display(comparison)

## 10. Freeze the Transformer reference, metadata, and checksums

In [ ]:
if PRIMARY_FEATURE_SET not in set(results["feature_set"].astype(str)):
    raise ValueError(
        f"Primary feature set {PRIMARY_FEATURE_SET!r} is required to freeze the Transformer reference."
    )

primary_result = results.loc[results["feature_set"].eq(PRIMARY_FEATURE_SET)].iloc[0]
transformer_reference = {
    "model_version": MODEL_VERSION,
    "model": "causal_temporal_transformer",
    "feature_set": PRIMARY_FEATURE_SET,
    "target_definition": target_definition,
    "sequence_length": SEQUENCE_LENGTH,
    "selected_threshold": optional_float(primary_result["selected_threshold"]),
    "validation_auprc": optional_float(primary_result["validation_auprc"]),
    "test_auprc": optional_float(primary_result["test_auprc"]),
    "test_f1": optional_float(primary_result["test_f1"]),
    "test_recall": optional_float(primary_result["test_recall"]),
    "latency_p95_ms": optional_float(primary_result["latency_p95_ms"]),
    "hypothesis_h2_r_gcn_margin_required": 0.03,
    "future_comparison_rule": "R-GCN test AUPRC minus this Transformer test AUPRC",
    "interpretation": (
        "Controlled cross-domain alignment performance; not evidence of a site-coincident or causal "
        "aircraft-cyber relationship."
    ),
}
REFERENCE_OUT.write_text(json.dumps(transformer_reference, indent=2, default=str), encoding="utf-8")

configuration_payload = {
    "model_version": MODEL_VERSION,
    "random_state": RANDOM_STATE,
    "feature_sets_to_run": FEATURE_SETS_TO_RUN,
    "primary_feature_set": PRIMARY_FEATURE_SET,
    "training_configuration": asdict(training_configuration),
    "device": str(DEVICE),
    "time_gap_standardization": {
        "mean_log1p_training": TIME_GAP_MEAN,
        "std_log1p_training": TIME_GAP_STD,
    },
    "sequence_policy": {
        "stream": "global_site_chronological_event_stream",
        "includes_current_event": True,
        "maximum_sequence_length": SEQUENCE_LENGTH,
        "padding": "left_zero_padding",
        "causal_mask": True,
        "future_events_allowed": False,
        "prior_partition_predictors_allowed": True,
        "target_labels_in_sequence": False,
    },
}
CONFIG_OUT.write_text(json.dumps(configuration_payload, indent=2, default=str), encoding="utf-8")

input_checksums = {
    "feature_bank": sha256_file(FEATURES_IN),
    "split_assignments": sha256_file(SPLITS_IN),
    "feature_manifest": sha256_file(FEATURE_MANIFEST_IN),
    "target_definition": sha256_file(TARGET_DEFINITION_IN),
}

regression_checks = [
    {
        "metric": "feature_rows_equal_split_rows",
        "expected": len(features),
        "actual": len(model_data),
        "match": len(features) == len(model_data),
    },
    {
        "metric": "chronological_training_has_both_classes",
        "expected": True,
        "actual": has_both_classes(model_data.loc[model_data[SPLIT_COLUMN].eq("train"), TARGET_COLUMN]),
        "match": has_both_classes(model_data.loc[model_data[SPLIT_COLUMN].eq("train"), TARGET_COLUMN]),
    },
    {
        "metric": "all_sequences_end_at_current_event",
        "expected": True,
        "actual": bool((sequence_manifest["sequence_end_position"] == np.arange(len(model_data))).all()),
        "match": bool((sequence_manifest["sequence_end_position"] == np.arange(len(model_data))).all()),
    },
    {
        "metric": "all_sequence_starts_not_after_current_time",
        "expected": True,
        "actual": bool(
            (
                pd.to_datetime(sequence_manifest["sequence_start_time_utc"], utc=True)
                <= pd.to_datetime(sequence_manifest["current_event_time_utc"], utc=True)
            ).all()
        ),
        "match": bool(
            (
                pd.to_datetime(sequence_manifest["sequence_start_time_utc"], utc=True)
                <= pd.to_datetime(sequence_manifest["current_event_time_utc"], utc=True)
            ).all()
        ),
    },
    {
        "metric": "primary_transformer_result_rows",
        "expected": 1,
        "actual": int(results["feature_set"].eq(PRIMARY_FEATURE_SET).sum()),
        "match": int(results["feature_set"].eq(PRIMARY_FEATURE_SET).sum()) == 1,
    },
    {
        "metric": "test_prediction_rows_for_each_model",
        "expected": int(model_data[SPLIT_COLUMN].eq("test").sum()) * len(FEATURE_SETS_TO_RUN),
        "actual": int(prediction_table["split"].eq("test").sum()),
        "match": int(prediction_table["split"].eq("test").sum())
        == int(model_data[SPLIT_COLUMN].eq("test").sum()) * len(FEATURE_SETS_TO_RUN),
    },
    {
        "metric": "primary_p95_latency_under_500ms",
        "expected": True,
        "actual": optional_float(primary_result["latency_p95_ms"]) <= 500.0,
        "match": optional_float(primary_result["latency_p95_ms"]) <= 500.0,
    },
]

summary = {
    "definitions": {
        "model_version": MODEL_VERSION,
        "primary_feature_set": PRIMARY_FEATURE_SET,
        "feature_sets_run": FEATURE_SETS_TO_RUN,
        "target_definition": target_definition,
        "sequence_length": SEQUENCE_LENGTH,
        "architecture": asdict(training_configuration),
    },
    "interpretation": {
        "site_coincident_claim": False,
        "causal_claim": False,
        "statement": (
            "The Transformer is a controlled temporal comparator using Notebook 03 alignment outputs. "
            "It does not establish real simultaneous PANC aircraft and network events."
        ),
    },
    "inputs": {
        "event_rows": len(model_data),
        "unique_aircraft_keys": int(model_data[GROUP_COLUMN].nunique()),
        "event_time_start_utc": model_data[TIME_COLUMN].min().isoformat(),
        "event_time_end_utc": model_data[TIME_COLUMN].max().isoformat(),
        "split_counts": model_data[SPLIT_COLUMN].value_counts().to_dict(),
        "positive_targets": int(model_data[TARGET_COLUMN].sum()),
        "target_prevalence": float(model_data[TARGET_COLUMN].mean()),
        "input_sha256": input_checksums,
    },
    "outputs": {
        "model_runs": len(results),
        "prediction_rows": len(prediction_table),
        "history_rows": len(training_history),
        "feature_attribution_rows": len(feature_attribution),
        "attention_summary_rows": len(attention_summary),
        "saved_model_files": len(list(MODELS_DIR.glob("*.pt"))),
        "saved_preprocessor_files": len(list(PREPROCESSING_DIR.glob("*.joblib"))),
        "saved_figure_files": len(list(FIGURES_DIR.glob("*.png"))),
        "transformer_reference": transformer_reference,
    },
    "environment": {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "torch": torch.__version__,
        "device": str(DEVICE),
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
    "regression_checks": regression_checks,
}
SUMMARY_OUT.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")

artifact_descriptions = {
    RESULTS_OUT.name: "Transformer validation/test metrics, selected threshold, training time, and latency.",
    PREDICTIONS_OUT.name: "Event-level validation and test probabilities and predicted labels.",
    THRESHOLDS_OUT.name: "Validation threshold-selection policy and value for each feature set.",
    HISTORY_OUT.name: "Per-epoch training loss, validation loss, and validation AUPRC.",
    LATENCY_OUT.name: "End-to-end raw-sequence preprocessing and model-forward latency.",
    SEQUENCE_MANIFEST_OUT.name: "Event-level backward-looking sequence boundaries and available context.",
    FEATURE_ATTRIBUTION_OUT.name: "Gradient-times-input diagnostic attribution for transformed fused features.",
    ATTENTION_SUMMARY_OUT.name: "Final-layer final-token attention summarized by temporal position and target class.",
    COMPARISON_OUT.name: "Primary Transformer test AUPRC compared with the best Notebook 04 fused tabular baseline.",
    REFERENCE_OUT.name: "Frozen Transformer performance reference for the later R-GCN hypothesis test.",
    CONFIG_OUT.name: "Frozen architecture, sequence policy, feature sets, device, and time-gap scaling.",
    SUMMARY_OUT.name: "Build metadata, provenance, environment, output counts, and regression checks.",
}
data_dictionary = pd.DataFrame(
    [
        {"artifact": artifact, "artifact_role": description}
        for artifact, description in artifact_descriptions.items()
    ]
)
data_dictionary.to_csv(DATA_DICTIONARY_OUT, index=False)

artifact_paths = [
    RESULTS_OUT,
    PREDICTIONS_OUT,
    THRESHOLDS_OUT,
    HISTORY_OUT,
    LATENCY_OUT,
    SEQUENCE_MANIFEST_OUT,
    FEATURE_ATTRIBUTION_OUT,
    ATTENTION_SUMMARY_OUT,
    COMPARISON_OUT,
    REFERENCE_OUT,
    CONFIG_OUT,
    SUMMARY_OUT,
    DATA_DICTIONARY_OUT,
]
artifact_paths.extend(sorted(MODELS_DIR.glob("*.pt")))
artifact_paths.extend(sorted(PREPROCESSING_DIR.glob("*.joblib")))
artifact_paths.extend(sorted(FIGURES_DIR.glob("*.png")))

checksum_lines: list[str] = []
for artifact_path in artifact_paths:
    if artifact_path.is_file() and artifact_path != CHECKSUMS_OUT:
        checksum_lines.append(
            f"{sha256_file(artifact_path)}  {artifact_path.relative_to(OUTPUT_DIR).as_posix()}"
        )
CHECKSUMS_OUT.write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")

final_checks = pd.DataFrame(regression_checks)
display(final_checks)
if not final_checks.loc[~final_checks["metric"].eq("primary_p95_latency_under_500ms"), "match"].all():
    raise AssertionError("One or more structural regression checks failed.")
if not bool(final_checks.loc[final_checks["metric"].eq("primary_p95_latency_under_500ms"), "match"].iloc[0]):
    warnings.warn(
        "The primary Transformer p95 latency exceeded 500 ms on this environment. "
        "The result was retained for transparent hypothesis evaluation.",
        RuntimeWarning,
    )

print("\nNotebook 05 completed successfully.")
print(f"Primary test AUPRC:  {transformer_reference['test_auprc']}")
print(f"Primary test F1:     {transformer_reference['test_f1']}")
print(f"Primary p95 latency: {transformer_reference['latency_p95_ms']:.3f} ms")
print(f"Output directory:    {OUTPUT_DIR}")

## Output interpretation and next stage

The frozen `transformer_reference_v2.json` is the comparison point for the later R-GCN experiment. Under the current hypothesis rule, the R-GCN must exceed the fused Transformer test AUPRC by at least **0.03**.

The next notebook should construct the relational graph while preserving the same event target and chronological split:

```text
06_build_relational_graph_v2.ipynb
```

The subsequent model notebook can then train and evaluate the R-GCN against the reference frozen here.